# 16 · Intervals

> Quasi tutti i problemi su intervalli partono da un **ordinamento** (per inizio o per fine) e poi scorrono una volta confrontando l'intervallo corrente con l'ultimo tenuto. Le varianti più difficili aggiungono un heap.
> Idee per i singoli problemi: [Pattern algoritmici › Intervalli](../Fundamentals/Pattern%20algoritmici.md#intervalli)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Insert Interval | tre fasi: prima, sovrapposti (unisci), dopo | `while i < n and ...`, `min`/`max` |
| Merge Intervals | ordina per inizio, estendi l'ultimo se si sovrappone | `intervals.sort(key=lambda x: x[0])`, `merged[-1][1] = max(...)` |
| Non-overlapping Intervals | ordina per **fine**, tieni quello che finisce prima | `key=lambda x: x[1]`, contatore delle rimozioni |
| Meeting Rooms | ordina e controlla adiacenti | `zip(xs, xs[1:])`, oggetti `Interval` con `.start`/`.end` |
| Meeting Rooms II | quante riunioni attive al massimo | min-heap delle fini, oppure inizi e fini ordinati separatamente |
| Minimum Interval to Include Each Query | query ordinate + heap `(lunghezza, fine)` | `sorted(enumerate(queries), key=...)`, risultato per indice originale |

## Sintassi al volo

| Serve | Sintassi |
|---|---|
| ordina per inizio | `intervals.sort()` (le liste si confrontano elemento per elemento) |
| ordina per fine | `intervals.sort(key=lambda iv: iv[1])` |
| unpacking nel ciclo | `for start, end in intervals:` |
| si sovrappongono? | `a_start <= b_end and b_start <= a_end` (oppure `<` se gli estremi che si toccano non contano) |
| unione | `[min(a_start, b_start), max(a_end, b_end)]` |
| coppie adiacenti | `for prev, cur in zip(xs, xs[1:])` · `itertools.pairwise(xs)` |
| ordinare tenendo l'indice originale | `sorted(range(len(q)), key=lambda i: q[i])` |

## Sovrapposizione

Due intervalli **non** si sovrappongono se uno finisce prima che l'altro inizi. La sovrapposizione è la negazione: più facile da ricordare così. Controlla nel testo se gli estremi che si toccano (`[1, 2]` e `[2, 3]`) contano come sovrapposti.

In [1]:
def overlaps(a: list[int], b: list[int]) -> bool:
    return not (a[1] < b[0] or b[1] < a[0])  # = a[0] <= b[1] and b[0] <= a[1]

print(overlaps([1, 3], [2, 6]), overlaps([1, 2], [2, 3]), overlaps([1, 2], [3, 4]))  # -> True True False

True True False


## Ordina e unisci

Ordinati per inizio, un intervallo può sovrapporsi solo all'**ultimo** unito: confronti `start` con `merged[-1][1]`. Nota che `merged[-1]` è una lista: modificarla cambia il risultato direttamente.

In [2]:
slots = [[8, 10], [1, 3], [2, 6], [15, 18]]
slots.sort()
print(slots)  # -> [[1, 3], [2, 6], [8, 10], [15, 18]]

merged = [slots[0][:]]  # copia: non modificare l'input per errore
for start, end in slots[1:]:
    if start <= merged[-1][1]:
        merged[-1][1] = max(merged[-1][1], end)  # max: [1, 10] e [2, 3] → resta 10
    else:
        merged.append([start, end])
print(merged)  # -> [[1, 6], [8, 10], [15, 18]]

[[1, 3], [2, 6], [8, 10], [15, 18]]
[[1, 6], [8, 10], [15, 18]]


## Ordinare per fine (scelta greedy)

Per tenere il **massimo numero** di intervalli disgiunti (e quindi rimuoverne il minimo), ordina per fine e tieni sempre quello che finisce prima: lascia più spazio ai successivi.

In [3]:
bookings = [[1, 2], [2, 3], [3, 4], [1, 3]]
bookings.sort(key=lambda iv: iv[1])
print(bookings)  # -> [[1, 2], [2, 3], [1, 3], [3, 4]]  (sort stabile: a parità di fine resta l'ordine)

prev_end = float("-inf")
kept = 0
for start, end in bookings:
    if start >= prev_end:  # qui gli estremi che si toccano NON si sovrappongono
        kept += 1
        prev_end = end
print(kept, len(bookings) - kept)  # -> 3 1

[[1, 2], [2, 3], [1, 3], [3, 4]]
3 1


## Classi `Interval` (LintCode / NeetCode)

Nei problemi *Meeting Rooms* gli intervalli sono spesso oggetti con attributi invece di liste. Cambia solo l'accesso.

In [4]:
class Interval:
    def __init__(self, start: int, end: int):
        self.start, self.end = start, end

meetings = [Interval(5, 10), Interval(0, 30), Interval(15, 20)]
meetings.sort(key=lambda m: m.start)
print([(m.start, m.end) for m in meetings])  # -> [(0, 30), (5, 10), (15, 20)]
print(any(prev.end > cur.start for prev, cur in zip(meetings, meetings[1:])))  # -> True: conflitto

[(0, 30), (5, 10), (15, 20)]
True


## Quanti attivi contemporaneamente

**Heap delle fini:** ordina per inizio; per ogni riunione, libera le sale finite (`heap[0] <= start`), poi occupane una. La dimensione massima dell'heap è la risposta.

**Sweep line:** ordina separatamente inizi e fini, e scorri con due puntatori: un inizio fa `+1`, una fine `-1`.

In [5]:
import heapq

meetings = [(0, 30), (5, 10), (15, 20)]
ends: list[int] = []
peak = 0
for start, end in sorted(meetings):
    while ends and ends[0] <= start:
        heapq.heappop(ends)  # sala liberata
    heapq.heappush(ends, end)
    peak = max(peak, len(ends))
print(peak)  # -> 2

starts = sorted(s for s, _ in meetings)
finishes = sorted(e for _, e in meetings)
print(starts, finishes)  # -> [0, 5, 15] [10, 20, 30]

2
[0, 5, 15] [10, 20, 30]


## Query offline: ordina anche le query

Quando ci sono molte query, ordinale (insieme agli intervalli) e scorri entrambi una volta sola. Il risultato va però restituito nell'**ordine originale**: ordina gli indici, non i valori.

In [6]:
import heapq

intervals = sorted([[1, 4], [2, 4], [3, 6], [4, 4]])
queries = [5, 2, 3]
order = sorted(range(len(queries)), key=lambda i: queries[i])
print(order)  # -> [1, 2, 0]: prima la query 2, poi 3, poi 5

answer = [-1] * len(queries)
heap: list[tuple[int, int]] = []  # (lunghezza, fine)
i = 0
for qi in order:
    q = queries[qi]
    while i < len(intervals) and intervals[i][0] <= q:  # entrano gli intervalli iniziati
        start, end = intervals[i]
        heapq.heappush(heap, (end - start + 1, end))
        i += 1
    while heap and heap[0][1] < q:  # escono quelli già finiti
        heapq.heappop(heap)
    answer[qi] = heap[0][0] if heap else -1
print(answer)  # -> [4, 3, 3]

[1, 2, 0]
[4, 3, 3]


## Errori comuni

- Dimenticare di ordinare (o ordinare per il campo sbagliato).
- Unione con `merged[-1][1] = end` invece di `max(...)`: un intervallo contenuto accorcia il risultato.
- `<` contro `<=`: leggi se `[1, 2]` e `[2, 3]` sono in conflitto.
- Insert Interval: l'input è già ordinato, non serve riordinare; gestisci l'inserimento in testa e in coda.
- Restituire i risultati delle query nell'ordine ordinato invece che originale.